# Residual hedging policies and classical baselines

Train the Layer-1 residual policies for the four test regimes, select the
CVaR coefficient on validation data and evaluate the classical baselines.
These experiments supply Figure 3 and Table 1 of the paper.

Start from the experiment directory described in
[the reproduction guide](../docs/reproduction.md#training-from-scratch) and run all
cells in order. The panel uses three values of kappa and five seeds per regime.
Outputs are written under `new-reward/checkpoints/unified_v3/` and
`new-reward/checkpoints/baselines_v3/`.


## Environment


In [ ]:
# CTC local runtime
from pathlib import Path
import os, sys, json, time, math, random, glob, hashlib, importlib, shutil, copy
from itertools import product
from scipy import stats
from collections import deque
def _find_repository_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'ctc_runtime.py').is_file():
            return candidate
    raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

KIT_ROOT = _find_repository_root()
if str(KIT_ROOT) not in sys.path:
    sys.path.insert(0, str(KIT_ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import norm
from IPython.display import display
from ctc_runtime import initialize, generate_rollout

globals().update(initialize())
os.chdir(LOCAL_REPO)
DEVICE = torch.device('cpu')
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
print('Repository:', KIT_ROOT)
print('Device:', DEVICE, '| torch:', torch.__version__, '| threads:', torch.get_num_threads())


## Train, validation and test years


In [ ]:
# ── REGIME config (mixed multi-year training version) ──
#  mixed training: TRAIN_YEARS can be"long and diverse"year list(including bull/bear/crash),
#  so g1 learns to control the tail across all regimes. But it must strictly satisfy: every train year < valyear < testyear.

# ── preset: uncomment one of these lines (crash panel: run first crash_test2018, then change to crash_test2020 rerun the fullnotebook) ──
REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'crash_test2018', list(range(2008,2017)), [2017], [2018]    # clean OOS crash: Feb Volmageddon + Q4 sharp drop
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'crash_test2020', list(range(2008,2019)), [2019], [2020]   # clean OOS crash: COVID (full year + crash slice)
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'mix_test2022', list(range(2008,2021)), [2021], [2022]      # mixed 13 years → bear market test
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'mix_test2024', list(range(2008,2023)), [2023], [2024]      # mixed → bull market test
# REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = 'narrow_test2022', [2019,2020], [2021], [2022]              # narrow-training control

# ── strictly guard against look-ahead (hard assertions): every train year < every val year < every test year ──
assert max(TRAIN_YEARS) < min(VAL_YEARS),  f'LOOK-AHEAD! train latest year {max(TRAIN_YEARS)} no earlier than val {min(VAL_YEARS)}'
assert max(VAL_YEARS)   < min(TEST_YEARS), f'LOOK-AHEAD! val latest year {max(VAL_YEARS)} no earlier than test {min(TEST_YEARS)}'
assert not (set(TRAIN_YEARS) | set(VAL_YEARS)) & set(TEST_YEARS), 'test year must not appear in train/val'

# ── crash sub-window: in"full year test", and within it separately evaluate the true crash period (verdict cell use) ──
#    decision rule: episode live interval [date_start, date_end] with any crash window"overlaps"counts it in,
#    because if the holding period touches the crash at all, this hedge has lived through the crash.
CRASH_WINDOWS = {
    'crash_test2018': [('2018-02-01', '2018-02-12'),   # Volmageddon (XIV blow-up, VIX doubled)
                       ('2018-10-01', '2018-12-26')],  # Q4 selloff all the way to the Christmas low
    'crash_test2020': [('2020-02-19', '2020-04-07')],  # COVID crash (top -> 3/23 bottom -> initial rebound)
}
CRASH_SLICES = CRASH_WINDOWS.get(REGIME, [])

OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{REGIME}'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(CLEAN_PARQUET), f'missing parquet: {CLEAN_PARQUET}'
print(f'REGIME={REGIME}')
print(f'  TRAIN={list(TRAIN_YEARS)}  (all < val)')
print(f'  VAL  ={VAL_YEARS}  (< test)')
print(f'  TEST ={TEST_YEARS}')
if CRASH_SLICES: print(f'  CRASH sub-slice={CRASH_SLICES}')
print('✓ strictly forward: max(train) < min(val) <= max(val) < min(test) — no look-ahead')


## Construct the data splits


In [ ]:
# ── split years by REGIME train/val/test ──
df_all = pd.read_parquet(CLEAN_PARQUET)
df_all['date'] = pd.to_datetime(df_all['date'])
yr = df_all['date'].dt.year
OUT_COLS = ['quote_datetime','ticker','expiration','strike',
            'underlying_bid','underlying_ask','bid','ask','nbr_next_steps']
for name, years in [('train', TRAIN_YEARS), ('validation', VAL_YEARS), ('test', TEST_YEARS)]:
    sub = df_all.loc[yr.isin(years)].sort_values(['expiration','strike','date']).reset_index(drop=True)
    g = sub.groupby(['expiration','strike'], sort=False)
    sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
    sub[OUT_COLS].to_csv(f'data/{name}.csv', index=False)
    print(f'  {name:11s}: years={years}  rows={len(sub):>9,}')
rates = (df_all[['date','r']].drop_duplicates('date').sort_values('date')
         .rename(columns={'date':'Date','r':'1y'}))
rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
rates = pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date')
rates.to_csv('data/1yr_treasury.csv', index=False)
print('treasury OK')


## Black–Scholes Greeks


In [ ]:
from scipy.stats import norm

def bs_greeks_inner(S, K, r, q, vol, T_years):
    if T_years <= 0 or vol <= 1e-8 or S <= 0 or K <= 0:
        return 0.0, 0.0, 0.0, 0.0
    sqT = np.sqrt(T_years)
    d1 = (np.log(S/K) + (r - q + 0.5*vol*vol) * T_years) / (vol * sqT)
    d2 = d1 - vol * sqT
    n_d1 = norm.pdf(d1)
    return (float(norm.cdf(d1)),
            float(n_d1 / (S * vol * sqT)),
            float(S * n_d1 * sqT),
            float(-(S * n_d1 * vol) / (2 * sqT) - r * K * np.exp(-r * T_years) * norm.cdf(d2)))


## Evaluation functions


In [ ]:
STATE_DIM, ACTION_DIM = 8, 1
TC_RATE = 0.001
REWARD_EXP = 1.0
GAMMA = 0.99
Q_DIV = 0.0
LAMBDAS_DEC = [0.0, 0.25, 0.5, 0.75, 1.0]
RATES = pd.read_csv('data/1yr_treasury.csv')
RATES['Date'] = pd.to_datetime(RATES['Date'])

def get_r(date):
    rows = RATES[RATES['Date']==date]
    if len(rows): return float(rows['1y'].iloc[0])
    rows = RATES[RATES['Date']<=date].sort_values('Date')
    return float(rows['1y'].iloc[-1]) if len(rows) else 0.04

def build_state(row, position_before_action):
    T_days = float(row['T'])
    spot   = float(row['S-1'])
    K      = spot / float(row['S/K'])
    IV     = float(row['v'])
    date   = pd.to_datetime(row['Date'])
    r      = get_r(date)
    d, g, vega, theta = bs_greeks_inner(spot, K, r, Q_DIV, IV, max(T_days, 0)/365.0)
    return np.array([float(row['S/K']), T_days/30.0, position_before_action, IV,
                     d, g, vega, theta], dtype=np.float32)

def cvar_5(arr):
    arr = np.asarray(arr); n_tail = max(1, int(0.05 * len(arr)))
    return np.sort(arr)[:n_tail].mean()


## Generate episode paths


In [ ]:
SEARCH_KAPPA = 2
REP_NAME = 'bs_reference'

def _csv_years(path):
    return set(pd.to_datetime(pd.read_csv(path, usecols=['Date'])['Date']).dt.year.astype(int))

def _file_md5(p, chunk=1<<20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()[:12]

def gen_rollout(split):
    return generate_rollout(split, REGIME, LOCAL_REPO)

ROLLF = {split: gen_rollout(split) for split in ('train', 'validation', 'test')}

def _episodes_from_csv(p):
    csv = pd.read_csv(p); csv['Date'] = pd.to_datetime(csv['Date'])
    eps = []
    for eid, edf in csv.groupby('episode', sort=False):
        edf = edf.sort_values('gstep').reset_index(drop=True)
        eps.append({'episode': int(eid), 'date_start': edf['Date'].iloc[0], 'ep_step_df': edf})
    return pd.DataFrame(eps).sort_values('date_start').reset_index(drop=True)

def load_eps(split): return _episodes_from_csv(ROLLF[split])

for split in ROLLF:
    print(f'{split}: {len(load_eps(split))} episodes')


## Residual policy and mean–CVaR objective


In [ ]:
# ── differentiable episode replay + BS residual policy + Rockafellar–Uryasev CVaR training ──
import torch, torch.nn as nn
DEVICE = torch.device('cpu')

def pack_episodes(eps_df):
    """flatten several episodes into flat tensors, for differentiable replay (state use the BS position trajectory, decoupled from the policy)."""
    states, b_pos, S_old, S_new, P_old, P_new, ep_id, is_first = [], [], [], [], [], [], [], []
    for ei, (_, erow) in enumerate(eps_df.iterrows()):
        df = erow['ep_step_df']; prev_b = 0.0
        for t in range(len(df)):
            r = df.iloc[t]
            states.append(build_state(r, prev_b))
            b = float(r['B Pos'])
            b_pos.append(b); S_old.append(float(r['S-1'])); S_new.append(float(r['S0']))
            P_old.append(float(r['P-1'])); P_new.append(float(r['P0']))
            ep_id.append(ei); is_first.append(1.0 if t == 0 else 0.0)
            prev_b = b
    t = lambda x, dt=torch.float32: torch.tensor(np.array(x), dtype=dt, device=DEVICE)
    return dict(states=t(states), b_pos=t(b_pos), S_old=t(S_old), S_new=t(S_new),
                P_old=t(P_old), P_new=t(P_new), ep_id=t(ep_id, torch.long),
                is_first=t(is_first), n_ep=int(eps_df.shape[0]))

class ResidualPolicy(nn.Module):
    def __init__(self, dim=8, hid=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hid), nn.ReLU(),
                                 nn.Linear(hid, hid), nn.ReLU(), nn.Linear(hid, 1))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)  # initial≈BS
    def forward(self, s): return self.net(s).squeeze(-1)

def _seg_sum(vals, ep_id, n_ep):
    out = torch.zeros(n_ep, device=vals.device, dtype=vals.dtype)
    return out.index_add_(0, ep_id, vals)

def _episode_reward(pack, pos, kappa):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    q = pnl * 100.0
    step_r = (q - kappa * torch.relu(-q)) * 10.0          # new downside reward, differentiable (relu)
    return _seg_sum(step_r, pack['ep_id'], pack['n_ep'])

def _positions(pack, policy):
    adj = policy(pack['states'])
    return torch.clamp(pack['b_pos'] + adj, -1.5, 1.5)     # position = BS delta + residual (B Pos=-delta∈[-1,0])

def _train_residual_once(pack_tr, kappa, beta, alpha=0.05, epochs=400, lr=1e-3, seed=0,
                         init_std=0.01, warm_frac=0.25):
    """single training run: small random init of the last layer + pure mean for the first warm_frac(β=0) + β ramp linearly to the target (v9)."""
    torch.manual_seed(seed); np.random.seed(seed)
    policy = ResidualPolicy().to(DEVICE)
    torch.nn.init.normal_(policy.net[-1].weight, std=init_std)
    torch.nn.init.zeros_(policy.net[-1].bias)
    eta = torch.zeros(1, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam(list(policy.parameters()) + [eta], lr=lr)
    bs_r = _episode_reward(pack_tr, pack_tr['b_pos'], kappa).detach()
    for ep in range(epochs):
        bt = 0.0 if ep < warm_frac * epochs else \
             beta * min(1.0, (ep - warm_frac*epochs) / max(1.0, warm_frac*epochs))
        opt.zero_grad()
        delta = _episode_reward(pack_tr, _positions(pack_tr, policy), kappa) - bs_r
        cvar = eta - (1.0 / alpha) * torch.relu(eta - delta).mean()
        loss = -(delta.mean() + bt * cvar)
        loss.backward(); opt.step()
    return policy

def _adj_scale(policy, pack):
    with torch.no_grad():
        return policy(pack['states']).abs().mean().item()

COLLAPSE_TH, N_RETRY = 1e-3, 1   # collapse criterion(trainside) and restart count — report the trigger frequency in the paper's methods section

def train_residual(pack_tr, kappa, beta, alpha=0.05, epochs=400, lr=1e-3, seed=0, **kw):
    """v9 = warmup training + auto-restart on collapse (the criterion looks only at the train side |adj|, does not touch test; seed+1000 retry)."""
    policy = _train_residual_once(pack_tr, kappa, beta, alpha, epochs, lr, seed, **kw)
    a = _adj_scale(policy, pack_tr); tries = 0
    while a < COLLAPSE_TH and tries < N_RETRY:
        tries += 1
        print(f'  ↻ collapse-retry{tries}: κ={kappa} β={beta} seed {seed}→{seed+1000*tries} (|adj|={a:.5f})')
        p2 = _train_residual_once(pack_tr, kappa, beta, alpha, epochs, lr, seed + 1000*tries, **kw)
        a2 = _adj_scale(p2, pack_tr)
        if a2 > a: policy, a = p2, a2
    if a < COLLAPSE_TH:
        print(f'  ⚠ collapse persists: κ={kappa} β={beta} seed={seed} (|adj|={a:.5f})')
    return policy

@torch.no_grad()
def eval_delta(pack, policy, kappa):
    bs_r = _episode_reward(pack, pack['b_pos'], kappa)
    return (_episode_reward(pack, _positions(pack, policy), kappa) - bs_r).cpu().numpy()

@torch.no_grad()
def eval_rewards(pack, policy, kappa):
    r_pol = _episode_reward(pack, _positions(pack, policy), kappa).cpu().numpy()
    r_bs  = _episode_reward(pack, pack['b_pos'], kappa).cpu().numpy()
    return r_pol, r_bs

print('residual-CVaR machinery ready; device =', DEVICE)


## Check reward replay against the environment


In [ ]:
# ── GATE0: replay(pure BS) vs the generating environment, episode by episode ──
# Reconciles the differentiable-replay reward against the environment's own 'B Reward' records
# (kappa=2: the rollout was generated by the k2_s0 model under kappa=2 settings; the machinery's
# _episode_reward uses the same scaling as env.py). Prerequisite: run cells up to `## 6` only.
import pandas as pd, numpy as np

te_csv = pd.read_csv(ROLLF['test'])
env_by_id = te_csv.groupby('episode')['B Reward'].sum()       # per-episode BS-arm reward recorded by the env
eps_te = load_eps('test')
env_bs = env_by_id.loc[eps_te['episode'].values].to_numpy()   # aligned to the pack's episode order

pte = pack_episodes(eps_te)
rep_bs = _episode_reward(pte, pte['b_pos'], 2).detach().cpu().numpy()   # replay: pure BS, kappa=2

d = np.abs(rep_bs - env_bs)
scale = max(np.abs(env_bs).max(), 1e-12)
print(f'episodes={len(d)}  max|replay-env|={d.max():.3e}  (relative {d.max()/scale:.2e})')
assert d.max() < 1e-4 * max(1.0, scale), 'GATE0 FAIL: systematic replay/env mismatch - check TC timing / price columns / scaling'
print('GATE0 PASS - replay formula matches the generating environment episode-by-episode (float32 rounding)')

## Training diagnostics


In [ ]:
import hashlib as _ahl, inspect as _ains
def _audit(tag, pack=None):
    parts = [f'[audit:{tag}]']
    try:
        parts.append('RATES=' + _ahl.md5(open('data/1yr_treasury.csv','rb').read()).hexdigest()[:8]
                     + f'({len(RATES)})')
    except Exception: parts.append('RATES=?')
    for f in (train_residual, _train_residual_once, _episode_reward, _positions, build_state):
        parts.append(f.__name__[:6] + '=' + _ahl.md5(_ains.getsource(f).encode()).hexdigest()[:6])
    if pack is not None:
        parts.append('states=' + _ahl.md5(pack['states'].detach().cpu().numpy().tobytes()).hexdigest()[:10])
    import torch as _t
    parts.append(f'thr={_t.get_num_threads()}')
    print(' '.join(parts))
print('audit ready')

## Paired estimates and block bootstrap


In [ ]:
import numpy as np

def _ep_dates(eps_df):
    """each episode's start date (datetime64, pack order)."""
    return pd.to_datetime(eps_df['date_start']).dt.floor('D').to_numpy()

def cvar5(a):
    a = np.asarray(a, float); k = max(1, int(0.05 * len(a)))
    return float(np.sort(a)[:k].mean())

def full_stat_ci(x, dates, stat=np.mean, B=3000, block_days=10, seed=0):
    """main convention: full-sample statistics + date-based blocks bootstrap CI.
    resampling unit=trading-day blocks (same-day episodes enter/exit as a whole), block length block_days absorbs temporal correlation."""
    x = np.asarray(x, float)
    d_uniq, inv = np.unique(dates, return_inverse=True)
    groups = [np.where(inv == k)[0] for k in range(len(d_uniq))]
    rng = np.random.default_rng(seed)
    point = float(stat(x))
    n = len(d_uniq); nb = int(np.ceil(n / block_days))
    bs = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n, nb)
        di = np.concatenate([(s + np.arange(block_days)) % n for s in starts])[:n]
        sel = np.concatenate([groups[k] for k in di])
        bs[b] = stat(x[sel])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return point, float(lo), float(hi), float((bs <= 0).mean())

def collapsed_spaced(x, dates, min_days=7):
    """auxiliary convention: same-day mean collapse → ≥min_days greedy spacing. Deterministic, no ties."""
    x = np.asarray(x, float)
    d_uniq, inv = np.unique(dates, return_inverse=True)
    daily = np.array([x[inv == k].mean() for k in range(len(d_uniq))])
    sel, last = [], None
    for k in range(len(d_uniq)):
        if last is None or (d_uniq[k] - last) / np.timedelta64(1, 'D') >= min_days:
            sel.append(k); last = d_uniq[k]
    sel = np.array(sel, dtype=int)
    return d_uniq[sel], daily[sel]

def spaced_mean_ci(x, dates, min_days=7, B=3000, L=5, seed=0):
    _, v = collapsed_spaced(x, dates, min_days)
    rng = np.random.default_rng(seed)
    n = len(v); nb = int(np.ceil(n / L))
    bs = np.empty(B)
    for b in range(B):
        starts = rng.integers(0, n, nb)
        idx = np.concatenate([(s + np.arange(L)) % n for s in starts])[:n]
        bs[b] = v[idx].mean()
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return float(v.mean()), float(lo), float(hi), float((bs <= 0).mean()), n

def summarize_dp(dp, dates, tag=''):
    """one-line aggregate: main-convention trio + auxiliary convention. Returns a dict for the summary table."""
    m, lo, hi, p = full_stat_ci(dp, dates, np.mean)
    w, wl, wh, _ = full_stat_ci(dp, dates, lambda a: float((np.asarray(a) > 0).mean()))
    t, tl, th, _ = full_stat_ci(dp, dates, cvar5)
    sm, sl, sh, sp_, nsp = spaced_mean_ci(dp, dates)
    print(f'{tag} full sample: ΔPnL={m:+.2f}bp [{lo:+.2f},{hi:+.2f}] p_le0={p:.3f} | '
          f'win={w:.0%} [{wl:.0%},{wh:.0%}] | tailCVaR={t:+.2f} [{tl:+.2f},{th:+.2f}]')
    print(f'{tag} collapse by day×≥7days (n={nsp}): ΔPnL={sm:+.2f}bp [{sl:+.2f},{sh:+.2f}] p_le0={sp_:.3f}')
    return dict(full=m, full_lo=lo, full_hi=hi, full_p=p, win=w, win_lo=wl, win_hi=wh,
                tail=t, tail_lo=tl, tail_hi=th, sp=sm, sp_lo=sl, sp_hi=sh, sp_p=sp_,
                n_spaced=nsp)

print('✓ E0 unified estimator ready (full_stat_ci / collapsed_spaced / summarize_dp)')

## Switch between test regimes


In [ ]:
_REGIME_PRESETS = {
    'crash_test2018': (list(range(2008, 2017)), [2017], [2018]),
    'crash_test2020': (list(range(2008, 2019)), [2019], [2020]),
    'mix_test2022':   (list(range(2008, 2021)), [2021], [2022]),
    'mix_test2024':   (list(range(2008, 2023)), [2023], [2024]),
}

def switch_regime(regime):
    """switch the global context to the specified regime: years/split/cache incorporation/ROLLF/fingerprint validation/β*load."""
    global REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS, CRASH_SLICES, OUT_DIR, ROLLF
    global CFG_TAG, _SPLIT_YEARS, RATES, best_beta
    global ALPHA, EPOCHS, LR, KAPPAS, POLICY_SEEDS
    import hashlib, glob, shutil, json as _json

    ty, vy, te_y = _REGIME_PRESETS[regime]
    assert max(ty) < min(vy) and max(vy) < min(te_y), 'LOOK-AHEAD!'
    REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = regime, ty, vy, te_y
    CRASH_SLICES = CRASH_WINDOWS.get(regime, [])
    OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{regime}'
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f'━━ switch → {regime}: train {ty[0]}-{ty[-1]} | val {vy} | test {te_y} '
          f'| crashslice={bool(CRASH_SLICES)}')

    # 1) rewrite data/{train,validation,test}.csv + Treasury rate (and ## 2 same logic)
    df_all = pd.read_parquet(CLEAN_PARQUET)
    df_all['date'] = pd.to_datetime(df_all['date'])
    yr = df_all['date'].dt.year
    out_cols = ['quote_datetime', 'ticker', 'expiration', 'strike',
                'underlying_bid', 'underlying_ask', 'bid', 'ask', 'nbr_next_steps']
    for name, years in [('train', ty), ('validation', vy), ('test', te_y)]:
        sub = df_all.loc[yr.isin(years)].sort_values(
            ['expiration', 'strike', 'date']).reset_index(drop=True)
        g = sub.groupby(['expiration', 'strike'], sort=False)
        sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
        sub[out_cols].to_csv(f'data/{name}.csv', index=False)
    rates = (df_all[['date', 'r']].drop_duplicates('date').sort_values('date')
             .rename(columns={'date': 'Date', 'r': '1y'}))
    rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
    old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
    pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date').to_csv(
        'data/1yr_treasury.csv', index=False)
    RATES = pd.read_csv('data/1yr_treasury.csv'); RATES['Date'] = pd.to_datetime(RATES['Date'])
    del df_all

    # 2) recomputed for this regime CFG_TAG / _SPLIT_YEARS (fixes the batch-phase tag freeze issue)
    CFG_TAG = hashlib.md5(repr((sorted(TRAIN_YEARS), sorted(VAL_YEARS), sorted(TEST_YEARS),
                                REP_NAME)).encode()).hexdigest()[:8]
    _SPLIT_YEARS = {'train': set(map(int, TRAIN_YEARS)),
                    'validation': set(map(int, VAL_YEARS)),
                    'test': set(map(int, TEST_YEARS))}

    # 4) ROLLF + hard validation + fingerprint archived
    ROLLF = {sp: gen_rollout(sp) for sp in ['train', 'validation', 'test']}
    import torch as _th
    print(f'[env] torch={_th.__version__}  '
          f'dev={_th.cuda.get_device_name(0) if _th.cuda.is_available() else "CPU"}')
    print(f'[fp ] parquet md5={_file_md5(CLEAN_PARQUET)}')
    for sp in ['train', 'validation', 'test']:
        yrs = _csv_years(ROLLF[sp]); exp = _SPLIT_YEARS[sp]
        assert yrs <= exp, f'{sp}: year out of range {sorted(yrs - exp)}!'
        assert not (exp - yrs), f'{sp}: missing years {sorted(exp - yrs)}!'
        print(f'[fp ] {sp:11s} {min(yrs)}–{max(yrs)} ✓  md5={_file_md5(ROLLF[sp])}')

    # 5) constants + β* (reads the batch-phase on-disk json, does not rerun the search)
    ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
    KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
    with open(f'{OUT_DIR}/regime_best_beta.json') as f:
        best_beta = {int(k): float(v) for k, v in _json.load(f).items()}
    print(f'★ β* = {best_beta}   (from {OUT_DIR}/regime_best_beta.json)')
    print('Regime ready:', REGIME)

print('switch_regime() ready — usage: switch_regime("mix_test2022")')


## Layer-1 panel: four regimes, three kappa values and five seeds


In [ ]:
# ── PART B. panel main loop: context→βsearch(tolerance band)→15policy→E0aggregate→per-episode dp saved to disk ──
import json as _json, torch
BETA_GRID = [0.0, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
REGIMES = [('crash_test2018', list(range(2008, 2017)), [2017], [2018]),
           ('crash_test2020', list(range(2008, 2019)), [2019], [2020]),
           ('mix_test2022',   list(range(2008, 2021)), [2021], [2022]),
           ('mix_test2024',   list(range(2008, 2023)), [2023], [2024])]
UNI_DIR = f'{DRIVE_REPO}/checkpoints/unified_v3'; os.makedirs(UNI_DIR, exist_ok=True)

def _episode_pnl(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    return _seg_sum(pnl, pack['ep_id'], pack['n_ep'])

def _set_regime(regime, ty, vy, te_y):
    global REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS, CRASH_SLICES, OUT_DIR, ROLLF, RATES
    assert max(ty) < min(vy) and max(vy) < min(te_y), 'LOOK-AHEAD!'
    REGIME, TRAIN_YEARS, VAL_YEARS, TEST_YEARS = regime, ty, vy, te_y
    CRASH_SLICES = CRASH_WINDOWS.get(regime, [])
    OUT_DIR = f'{DRIVE_REPO}/checkpoints/regime_{regime}'; os.makedirs(OUT_DIR, exist_ok=True)
    df_all = pd.read_parquet(CLEAN_PARQUET); df_all['date'] = pd.to_datetime(df_all['date'])
    yr = df_all['date'].dt.year
    cols = ['quote_datetime', 'ticker', 'expiration', 'strike',
            'underlying_bid', 'underlying_ask', 'bid', 'ask', 'nbr_next_steps']
    for name, years in [('train', ty), ('validation', vy), ('test', te_y)]:
        sub = df_all.loc[yr.isin(years)].sort_values(['expiration', 'strike', 'date']).reset_index(drop=True)
        g = sub.groupby(['expiration', 'strike'], sort=False)
        sub['nbr_next_steps'] = (g['date'].transform('count') - g.cumcount() - 1).astype('int32')
        sub[cols].to_csv(f'data/{name}.csv', index=False)
    rates = (df_all[['date', 'r']].drop_duplicates('date').sort_values('date')
             .rename(columns={'date': 'Date', 'r': '1y'}))
    rates['Date'] = pd.to_datetime(rates['Date']).dt.strftime('%Y-%m-%d')
    old = pd.read_csv('data/1yr_treasury.csv') if os.path.exists('data/1yr_treasury.csv') else pd.DataFrame()
    pd.concat([old, rates]).drop_duplicates('Date').sort_values('Date').to_csv('data/1yr_treasury.csv', index=False)
    RATES = pd.read_csv('data/1yr_treasury.csv'); RATES['Date'] = pd.to_datetime(RATES['Date'])
    del df_all
    ROLLF = {sp: gen_rollout(sp) for sp in ['train', 'validation', 'test']}
    exp = {'train': set(map(int, ty)), 'validation': set(map(int, vy)), 'test': set(map(int, te_y))}
    for sp in ['train', 'validation', 'test']:
        yrs = _csv_years(ROLLF[sp])
        assert yrs <= exp[sp] and not (exp[sp] - yrs), f'{sp}: year check failed {sorted(yrs)}'
    print('[fp ] parquet=' + _file_md5(CLEAN_PARQUET) + '  '
          + '  '.join(f'{sp}={_file_md5(ROLLF[sp])}' for sp in ROLLF))

PANEL_ROWS = []
for RG, TY, VY, TEY in REGIMES:
    print(f'\n========== {RG} ==========')
    _set_regime(RG, TY, VY, TEY)
    ptr = pack_episodes(load_eps('train')); pva = pack_episodes(load_eps('validation'))
    _audit(f'B:{RG}', ptr)
    SEARCH = []
    for kappa in KAPPAS:
        for beta in BETA_GRID:
            pol = train_residual(ptr, kappa, beta, ALPHA, EPOCHS, LR, seed=0)
            d = eval_delta(pva, pol, kappa)
            SEARCH.append({'kappa': kappa, 'beta': beta,
                           'mean_va': float(d.mean()), 'cvar_va': cvar5(d)})
    SEARCH = pd.DataFrame(SEARCH); SEARCH.to_csv(f'{OUT_DIR}/regime_search.csv', index=False)
    best_beta = {}
    for kappa in KAPPAS:
        sub = SEARCH[SEARCH.kappa == kappa]; feas = sub[sub.cvar_va > 0]
        if len(feas):
            r = feas.sort_values('mean_va', ascending=False).iloc[0]
        else:
            cvb = sub['cvar_va'].max()
            r = (sub[sub['cvar_va'] >= cvb - 0.25 * abs(cvb)]
                 .sort_values('mean_va', ascending=False).iloc[0])
        best_beta[kappa] = float(r['beta'])
    _json.dump(best_beta, open(f'{OUT_DIR}/regime_best_beta.json', 'w'), indent=2)
    print('β* =', best_beta)
    eps_te = load_eps('test'); pte = pack_episodes(eps_te)
    dates = _ep_dates(eps_te)
    sp0 = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_te.iterrows()])
    bs_pnl = _episode_pnl(pte, pte['b_pos']).detach().cpu().numpy()
    for kappa in KAPPAS:
        dps, devs = [], []
        for sd in POLICY_SEEDS:
            pol = train_residual(ptr, kappa, best_beta[kappa], ALPHA, EPOCHS, LR, seed=sd)
            with torch.no_grad():
                pos = _positions(pte, pol)
                devs.append(float((pos - pte['b_pos']).abs().mean()))
            dps.append((_episode_pnl(pte, pos).detach().cpu().numpy() - bs_pnl) / sp0 * 1e4)
        dpm = np.mean(dps, axis=0)
        np.save(f'{UNI_DIR}/dp_{RG}_k{kappa}.npy', dpm)
        pd.DataFrame({'date_start': dates.astype('datetime64[D]').astype(str),
                      'dp_bp': dpm}).to_csv(f'{UNI_DIR}/perep_dp_{RG}_k{kappa}.csv', index=False)
        r = summarize_dp(dpm, dates, tag=f'[{RG} κ={kappa}]')
        if CRASH_SLICES:
            m = np.zeros(len(dpm), bool)
            for a, b in CRASH_SLICES:
                m |= (dates >= np.datetime64(a)) & (dates <= np.datetime64(b))
            cm, cl, ch, _ = full_stat_ci(dpm[m], dates[m], np.mean, block_days=3)
            r.update(crash_mean=cm, crash_lo=cl, crash_hi=ch, crash_n=int(m.sum()))
            print(f'   crash slice(n={int(m.sum())}, descriptive): ΔPnL={cm:+.2f}bp [{cl:+.2f},{ch:+.2f}]')
        order = np.argsort(dpm)[::-1]; tot = float(dpm.sum())
        r.update(regime=RG, kappa=kappa, beta_star=best_beta[kappa],
                 posdev_mean=float(np.mean(devs)),
                 top3_share=float(dpm[order[:3]].sum() / tot) if abs(tot) > 1e-12 else np.nan)
        PANEL_ROWS.append(r); print()
PANEL = pd.DataFrame(PANEL_ROWS)
PANEL.to_csv(f'{UNI_DIR}/panel_v3_summary.csv', index=False)
print('=== panel v3 (main=full sample + daily blocksbootstrap; auxiliary=collapse by day×≥7days) ===')
print(PANEL[['regime', 'kappa', 'beta_star', 'full', 'full_lo', 'full_hi', 'win',
             'tail', 'sp', 'sp_lo', 'sp_hi', 'posdev_mean', 'top3_share']]
      .round(3).to_string(index=False))

## Deployment comparison for 2022


In [ ]:
# ── PART C prepare: 2022 context + packs + constants ──
switch_regime('mix_test2022')
ALPHA, EPOCHS, LR = 0.05, 400, 1e-3
KAPPAS, POLICY_SEEDS = [1, 2, 3], [0, 1, 2, 3, 4]
EPS_TR, EPS_VA, EPS_TE = load_eps('train'), load_eps('validation'), load_eps('test')
pack_tr, pack_va, pack_te = (pack_episodes(EPS_TR), pack_episodes(EPS_VA), pack_episodes(EPS_TE))
_audit('C:prep', pack_tr)
BEST_BETA_BASE = dict(best_beta)
DATES_TE = _ep_dates(EPS_TE)
SP_TE = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in EPS_TE.iterrows()])
BS_PNL_TE = _episode_pnl(pack_te, pack_te['b_pos']).detach().cpu().numpy()
print('C context ready | β* =', BEST_BETA_BASE)

### Conformal controller


In [ ]:
# ── method3 v2: closed-loop Conformal Risk Control λ_t — the episode-level loss budget that controls the deployed policy ──
EXP_NAME = 'conformal_v3'
EXP_DIR  = f'{DRIVE_REPO}/checkpoints/{EXP_NAME}_{REGIME}'; os.makedirs(EXP_DIR, exist_ok=True)
LAM_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

def _episode_pnl(pack, pos):
    p_prev = torch.cat([torch.zeros(1, device=pos.device), pos[:-1]]) * (1.0 - pack['is_first'])
    tc  = -torch.abs(pos - p_prev) * pack['S_old'] * TC_RATE
    pnl = pos * (pack['S_new'] - pack['S_old']) + (pack['P_new'] - pack['P_old']) + tc
    return _seg_sum(pnl, pack['ep_id'], pack['n_ep'])

@torch.no_grad()
def episode_delta_lambda(pack, policy, kappa, lam_vec):
    """reward convention Δ (for evaluation, consistent with the v7 report)."""
    adj = policy(pack['states'])
    lam_step = torch.as_tensor(np.asarray(lam_vec, dtype=np.float32), device=DEVICE)[pack['ep_id']]
    pos = torch.clamp(pack['b_pos'] + lam_step * adj, -1.5, 1.5)
    r_pol = _episode_reward(pack, pos, kappa)
    r_bs  = _episode_reward(pack, pack['b_pos'], kappa)
    return (r_pol - r_bs).cpu().numpy(), r_pol.cpu().numpy(), r_bs.cpu().numpy()

@torch.no_grad()
def episode_dpnl_bp_lambda(pack, policy, lam_vec, start_spot):
    """PnL convention Δ, in bp of notional (for control/budget, κ-unrelated)."""
    adj = policy(pack['states'])
    lam_step = torch.as_tensor(np.asarray(lam_vec, dtype=np.float32), device=DEVICE)[pack['ep_id']]
    pos = torch.clamp(pack['b_pos'] + lam_step * adj, -1.5, 1.5)
    dp = (_episode_pnl(pack, pos) - _episode_pnl(pack, pack['b_pos'])).cpu().numpy()
    return dp / start_spot * 1e4

def conformal_lower_q(v, w, eps):
    """weighted conformal lower tail ε quantile, virtual point −∞ weight 1 (conservative correction). Insufficient samples → −inf."""
    v = np.asarray(v, float); w = np.asarray(w, float)
    W = w.sum() + 1.0
    if W <= 0 or 1.0 / W >= eps: return -np.inf
    idx = np.argsort(v); v, w = v[idx], w[idx]
    cw = (1.0 + np.cumsum(w)) / W          # −∞ mass entirely at the left end
    k = int(np.searchsorted(cw, eps, side='left'))
    return float(v[min(k, len(v) - 1)])

def precompute_grids(eps_df, pack, policy, kappa):
    """each (policy, split) computed only once: dates/спот/IV + all λ the grid's reward-Δ and PnL-Δbp."""
    n  = len(eps_df)
    ds = pd.to_datetime(eps_df['date_start']).to_numpy()
    de = np.array([pd.to_datetime(e['ep_step_df']['Date']).max()
                   for _, e in eps_df.iterrows()], dtype='datetime64[ns]')
    ss = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_df.iterrows()])
    iv = np.array([float(e['ep_step_df']['v'].iloc[0])  for _, e in eps_df.iterrows()])
    order = np.argsort(ds)
    d_lam  = {la: episode_delta_lambda(pack, policy, kappa, np.full(n, la))[0] for la in LAM_GRID}
    dp_lam = {la: episode_dpnl_bp_lambda(pack, policy, np.full(n, la), ss)     for la in LAM_GRID}
    return dict(n=n, ds=ds, de=de, ss=ss, iv=iv, order=order, d_lam=d_lam, dp_lam=dp_lam)

def calibrate_budget_bp(pack_va, eps_va_df, policy, eps_target, floor_bp=5.0):
    """budget c = validation fix1's ΔPnL_bp of ε absolute quantile value (lower bound floor_bp)."""
    ss = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_va_df.iterrows()])
    dp = episode_dpnl_bp_lambda(pack_va, policy, np.ones(len(eps_va_df)), ss)
    return float(max(floor_bp, -np.quantile(dp, eps_target)))

def deploy_lambda_control(eps_df, pack, policy, kappa, mode='crc_rwc', eps_target=0.10,
                          budget_bp=25.0, grids=None, gamma_eps=0.02, half_life=30.0,
                          tau_iv=0.04, warmup=15, M_emp=0.3, lam_warmup=1.0):
    """chronological episode-by-episode deployment λ_t.
       mode ∈ {fix1, fix0, a2margin, aci_open, crc, crc_rwc, crc_rwc_aci}"""
    g = grids if grids is not None else precompute_grids(eps_df, pack, policy, kappa)
    n, ds, de, iv, order = g['n'], g['ds'], g['de'], g['iv'], g['order']
    d_lam, dp_lam = g['d_lam'], g['dp_lam']
    used = np.full(n, np.nan); informed = np.zeros(n, bool)
    lam_aci = float(lam_warmup); eps_eff = float(eps_target)
    aci_pending = []                                   # crc_rwc_aci delayed-feedback queue
    for t, i in enumerate(order):
        past = order[:t]
        done = past[de[past] < ds[i]]                  # ★ use only completed episode (fix look-ahead)
        informed[i] = len(done) >= warmup
        # —— crc_rwc_aci: first digest"completed by now"deployment results, online-tune ε_eff (closed-loop + delayed feedback) ——
        if mode == 'crc_rwc_aci' and aci_pending:
            still = []
            for j in aci_pending:
                if de[j] < ds[i]:
                    err = float(dp_lam[used[j]][j] < -budget_bp)
                    eps_eff = float(np.clip(eps_eff + gamma_eps * (eps_target - err),
                                            0.25 * eps_target, min(0.45, 4 * eps_target)))
                else:
                    still.append(j)
            aci_pending = still
        # —— decision ——
        if   mode == 'fix1': lam_t = 1.0
        elif mode == 'fix0': lam_t = 0.0
        elif not informed[i]:
            lam_t = 0.0 if mode == 'a2margin' else lam_warmup
        elif mode == 'a2margin':                       # old A2-style (completion filtering fixed)
            thr = cvar_5(d_lam[0.0][done]) - M_emp
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):
                if cvar_5(d_lam[la][done]) >= thr: lam_t = la; break
        elif mode == 'aci_open':                       # old open-loop rule (ablation control, only fixes completion filtering)
            var_hat = np.quantile(dp_lam[1.0][done], eps_target)
            err = float(dp_lam[1.0][i] < var_hat)      # events remain aligned with deployment λ unrelated → open-loop
            lam_aci = float(np.clip(lam_aci + 0.10 * (eps_target - err), 0.0, 1.0))
            lam_t = lam_aci
        else:                                          # crc / crc_rwc / crc_rwc_aci (closed-loop)
            if mode in ('crc_rwc', 'crc_rwc_aci'):
                dt_ = (ds[i] - de[done]) / np.timedelta64(1, 'D')
                w = (np.exp(-np.log(2) * np.maximum(dt_, 0.0) / half_life)
                     * np.exp(-np.abs(iv[done] - iv[i]) / tau_iv))
            else:
                w = np.ones(len(done))
            e_use = eps_eff if mode == 'crc_rwc_aci' else eps_target
            lam_t = 0.0
            for la in sorted(LAM_GRID, reverse=True):  # risk varies with λ monotonically ⇒ take the largest feasible λ
                if conformal_lower_q(dp_lam[la][done], w, e_use) >= -budget_bp:
                    lam_t = la; break
        used[i] = lam_t
        if mode == 'crc_rwc_aci': aci_pending.append(i)
    # —— unified evaluation: reward convention (consistent with the v7 report) + post-deployment realized breach (all modes comparable) ——
    d_used, rp, rbs = episode_delta_lambda(pack, policy, kappa, used)
    dp_used = episode_dpnl_bp_lambda(pack, policy, used, g['ss'])
    m = informed
    breach = float(np.mean(dp_used[m] < -budget_bp)) if m.any() else np.nan
    return {'lam': used, 'd': d_used, 'rp': rp, 'rbs': rbs, 'dpnl_bp': dp_used,
            'breach': breach, 'informed': m, 'budget_bp': float(budget_bp),
            'eps_eff_final': float(eps_eff)}

print('closed-loop CRC machinery ready')

### Validation-selected policy and loss budget


In [ ]:
# ── C-baselinearm: 15policy + E0 aggregate + immediate reconciliation with the PART B dp ──
BASE = {}
for kappa in KAPPAS:
    rows, dps = [], []
    for sd in POLICY_SEEDS:
        pol = train_residual(pack_tr, kappa, BEST_BETA_BASE[kappa], ALPHA, EPOCHS, LR, seed=sd)
        rows.append({'seed': sd, 'pol': pol})
        dps.append((_episode_pnl(pack_te, _positions(pack_te, pol)).detach().cpu().numpy()
                    - BS_PNL_TE) / SP_TE * 1e4)
    BASE[kappa] = rows
    dpm = np.mean(dps, axis=0)
    ref = np.load(f'{UNI_DIR}/dp_mix_test2022_k{kappa}.npy')
    print(f'κ={kappa} byte-identical to PART B dp: {bool(np.array_equal(dpm, ref))}')
    summarize_dp(dpm, DATES_TE, tag=f'[C-base κ={kappa}]'); print()
# budget raw quantile (the paper reports c=max(5bp, q̂))
ssv = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in EPS_VA.iterrows()])
for kappa in KAPPAS:
    qs = []
    for i, sd in enumerate(POLICY_SEEDS):
        adj = BASE[kappa][i]['pol'](pack_va['states']).detach()
        import torch as _t
        pos = _t.clamp(pack_va['b_pos'] + adj, -1.5, 1.5)
        dpv = (_episode_pnl(pack_va, pos).detach().cpu().numpy()
               - _episode_pnl(pack_va, pack_va['b_pos']).detach().cpu().numpy()) / ssv * 1e4
        qs.append(-float(np.quantile(dpv, 0.10)))
    print(f'κ={kappa}: raw q̂_10% = ' + ', '.join(f'{q:+.2f}' for q in qs) + '  (floor=5.0bp)')

### Deployment rules


In [ ]:
# ── C-six-rule comparison (E0convention): fix1/a2margin/aci_open/crc/crc_rwc/crc_rwc_aci ──
MODES = ['fix1', 'a2margin', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']
EPS_T = 0.10
budgets = {(k, sd): calibrate_budget_bp(pack_va, EPS_VA, BASE[k][i]['pol'], EPS_T)
           for k in KAPPAS for i, sd in enumerate(POLICY_SEEDS)}
SIX_ROWS, SIX_PATH = [], {}
for kappa in KAPPAS:
    grids_per_seed = {}
    for i, sd in enumerate(POLICY_SEEDS):
        grids_per_seed[sd] = precompute_grids(EPS_TE, pack_te, BASE[kappa][i]['pol'], kappa)
    for mode in MODES:
        paths = []
        br, lam = [], []
        for i, sd in enumerate(POLICY_SEEDS):
            out = deploy_lambda_control(EPS_TE, pack_te, BASE[kappa][i]['pol'], kappa,
                                        mode=mode, eps_target=EPS_T,
                                        budget_bp=budgets[(kappa, sd)],
                                        grids=grids_per_seed[sd])
            paths.append(out['dpnl_bp']); br.append(out['breach']); lam.append(float(np.nanmean(out['lam'])))
        dpm = np.mean(paths, axis=0)
        m, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
        SIX_ROWS.append({'kappa': kappa, 'mode': mode, 'full': m, 'lo': lo, 'hi': hi,
                         'p_le0': p, 'breach': float(np.nanmean(br)), 'lam': float(np.mean(lam))})
        SIX_PATH[(kappa, mode)] = dpm
        print(f'κ={kappa} {mode:12s}: ΔPnL={m:+.2f}bp [{lo:+.2f},{hi:+.2f}] '
              f'breach={np.nanmean(br):.4f} λ̄={np.mean(lam):.3f}')
    f1 = SIX_PATH[(kappa, 'fix1')]
    for mode in MODES[1:]:
        ret = SIX_PATH[(kappa, mode)].mean() / f1.mean() * 100 if abs(f1.mean()) > 1e-12 else np.nan
        print(f'   retention {mode:12s} = {ret:.0f}%')
    dm, dl, dh, dp_ = full_stat_ci(SIX_PATH[(kappa, 'crc_rwc')] - f1, DATES_TE, np.mean)
    print(f'   crc_rwc−fix1: Δ={dm:+.2f}bp [{dl:+.2f},{dh:+.2f}] p_le0={dp_:.3f}\n')
SIX = pd.DataFrame(SIX_ROWS); SIX.to_csv(f'{EXP_DIR}/six_rules_{REGIME}.csv', index=False)

### Aggressive policy and retained improvement


In [ ]:
# ── C-aggressive arm β=0.5 → εfrontier → β=0 fire → enabler certify (all under the E0 convention) ──
def run_arm(beta_arm, tag):
    pols, grids = {}, {}
    for i, sd in enumerate(POLICY_SEEDS):
        for kappa in KAPPAS:
            if (kappa, sd) not in pols:
                p_ = train_residual(pack_tr, kappa, beta_arm, ALPHA, EPOCHS, LR, seed=sd)
                pols[(kappa, sd)] = p_
                grids[(kappa, sd)] = precompute_grids(EPS_TE, pack_te, p_, kappa)
    rows = {}
    for kappa in KAPPAS:
        for mode in ['fix1', 'aci_open', 'crc', 'crc_rwc', 'crc_rwc_aci']:
            paths, br, lam = [], [], []
            for i, sd in enumerate(POLICY_SEEDS):
                out = deploy_lambda_control(EPS_TE, pack_te, pols[(kappa, sd)], kappa,
                                            mode=mode, eps_target=EPS_T,
                                            budget_bp=budgets[(kappa, sd)], grids=grids[(kappa, sd)])
                paths.append(out['dpnl_bp']); br.append(out['breach']); lam.append(float(np.nanmean(out['lam'])))
            dpm = np.mean(paths, axis=0)
            m, lo, hi, p = full_stat_ci(dpm, DATES_TE, np.mean)
            t, tl, th, _ = full_stat_ci(dpm, DATES_TE, cvar5)
            rows[(kappa, mode)] = dict(dpm=dpm, m=m, lo=lo, hi=hi, p=p,
                                       breach=float(np.nanmean(br)), lam=float(np.mean(lam)),
                                       tail=t, tl=tl, th=th)
            print(f'{tag} κ={kappa} {mode:12s}: ΔPnL={m:+.2f} [{lo:+.2f},{hi:+.2f}] '
                  f'breach={np.nanmean(br):.3f} λ̄={np.mean(lam):.2f} tail={t:+.2f}[{tl:+.2f},{th:+.2f}]')
        f1 = rows[(kappa, 'fix1')]
        print(f'{tag} κ={kappa} prerequisite: fix1 breach={f1["breach"]:.3f} vs ε={EPS_T} → '
              + ('✓ firing scenario' if f1['breach'] > EPS_T else 'calm scenario'))
    return rows

print('━━ mildly aggressive arm β=0.5 ━━')
ARM05 = run_arm(0.5, '[β=0.5]')
print('\n━━ wild arm β=0 (fire) ━━')
ARM0 = run_arm(0.0, '[β=0]')
print('\n━━ enabler certify: guarded wild arm (crc_rwc_aci) vs base arm ━━')
for kappa in KAPPAS:
    g = ARM0[(kappa, 'crc_rwc_aci')]
    b = np.load(f'{UNI_DIR}/dp_mix_test2022_k{kappa}.npy')
    dm, dl, dh, dp_ = full_stat_ci(g['dpm'] - b, DATES_TE, np.mean)
    print(f'κ={kappa}: guarded wild arm={g["m"]:+.2f}bp [{g["lo"]:+.2f},{g["hi"]:+.2f}] breach={g["breach"]:.3f} | '
          f'vsbase arm Δ={dm:+.2f}bp [{dl:+.2f},{dh:+.2f}] p_le0={dp_:.3f}')
import pickle
with open(f'{EXP_DIR}/arms_{REGIME}.pkl', 'wb') as fh:
    pickle.dump({k: {kk: (vv if not isinstance(vv, np.ndarray) else vv.tolist())
                     for kk, vv in v.items()} for k, v in
                 {**{('05',)+k: v for k, v in ARM05.items()},
                  **{('00',)+k: v for k, v in ARM0.items()}}.items()}, fh)

## Classical baselines


In [ ]:
# ── D1. classic baseline mechanism: corrected version HW-MV (time seriesβ) + Whalley-Wilmott no-trade band ──
# design highlights:
#  · HW-MV by Hull-White(2017) original version: use **historical time series** regression Δσ_imp = (a+bδ+cδ²)·ΔS/(S√T),
#    estimation window = the current regime's train+val years (strictly earlier than the test year, no look-ahead).
#    —— the old shield implementation using the cross-sectional smile slope was directionally wrong, this is the corrected version.
#  · WW with: position stays within [−δ−h, −δ+h] within, h=(1.5·tc·S·Γ²/γ)^(1/3);
#    risk aversion γ on the validation year by κ-reward select (same protocol as the residual, preregistered grid).
#  · paired residual arm: κ=2 (middle tier, preregistered), 5 seeds deterministic retraining = same policies as the panel.
WW_GAMMA_GRID = [0.05, 0.1, 0.5, 1.0, 5.0, 20.0]   # v3: all aggregation uses the E0 estimator

def _replay_pos(eps_df, pos_fn, kappa):
    """replay any position sequence episode by episode → (pnl, κ-reward) array. Consistent with cell14's PnL/cost calculation."""
    pnls, rews = [], []
    for _, erow in eps_df.iterrows():
        df = erow['ep_step_df']; tgt = pos_fn(df)
        pos_prev, pnl, rew = 0.0, 0.0, 0.0
        for t in range(len(df)):
            r = df.iloc[t]; p = float(tgt[t])
            tc = -abs(p - pos_prev) * float(r['S-1']) * TC_RATE
            sp = p * (float(r['S0']) - float(r['S-1'])) + (float(r['P0']) - float(r['P-1'])) + tc
            q = sp * 100.0
            rew += (q - kappa * max(0.0, -q)) * 10.0
            pnl += sp; pos_prev = p
        pnls.append(pnl); rews.append(rew)
    return np.array(pnls), np.array(rews)

def _pos_bs(df):
    return df['B Pos'].astype(float).values

def _greeks_row(r):
    S = float(r['S-1']); K = S / float(r['S/K']); iv = float(r['v'])
    Ty = max(float(r['T']), 1e-6) / 365.0
    rr = get_r(pd.to_datetime(r['Date']))
    return S, bs_greeks_inner(S, K, rr, Q_DIV, iv, Ty), Ty

def make_hw_fn(a, b, c):
    def f(df):
        out = []
        for t in range(len(df)):
            S, (d, g, vega, _), Ty = _greeks_row(df.iloc[t])
            dmv = d + vega * (a + b * d + c * d * d) / (S * np.sqrt(max(Ty, 1e-6)))
            out.append(-float(np.clip(dmv, 0.0, 1.0)))
        return np.array(out)
    return f

def make_ww_fn(gam):
    def f(df):
        pos, out = 0.0, []
        for t in range(len(df)):
            S, (d, g, _, _), _ = _greeks_row(df.iloc[t])
            tgt = -d
            h = (1.5 * TC_RATE * S * g * g / gam) ** (1.0 / 3.0)
            if   pos < tgt - h: pos = tgt - h
            elif pos > tgt + h: pos = tgt + h
            out.append(pos)
        return np.array(out)
    return f

def estimate_hw_ts(years):
    """time series β estimate: Δσ = (a+bδ+cδ²)·ΔS/(S√T) + ε, use only the given years (≤val)."""
    cols = ['date', 'expiration', 'strike', 'impl_volatility',
            'underlying_bid', 'underlying_ask', 'r']
    df = pd.read_parquet(CLEAN_PARQUET, columns=cols)
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'].dt.year.isin(set(years))]
    df['S'] = (df['underlying_bid'] + df['underlying_ask']) / 2
    df = df.dropna(subset=['impl_volatility', 'S'])
    df['T'] = (pd.to_datetime(df['expiration']) - df['date']).dt.days / 365.0
    df = df[(df['T'] > 7 / 365) & (df['T'] < 1.01)]
    df = df.sort_values(['expiration', 'strike', 'date'])
    g = df.groupby(['expiration', 'strike'], sort=False)
    df['dS'] = g['S'].diff(); df['dIV'] = g['impl_volatility'].diff()
    df['gap'] = g['date'].diff().dt.days
    df = df.dropna(subset=['dS', 'dIV'])
    df = df[(df['gap'] <= 3) & (np.abs(df['dS'] / df['S']) < 0.10)]
    sig = df['impl_volatility'].clip(0.01, 3.0)
    rr = df['r'].fillna(0.02) if 'r' in df else 0.02
    d1 = (np.log(df['S'] / df['strike']) + (rr + 0.5 * sig**2) * df['T']) / (sig * np.sqrt(df['T']))
    df['delta'] = norm.cdf(d1)
    df = df[(df['delta'] > 0.05) & (df['delta'] < 0.95)]
    if len(df) > 400_000: df = df.sample(400_000, random_state=0)
    x = (df['dS'] / (df['S'] * np.sqrt(df['T']))).values
    X = np.column_stack([x, x * df['delta'].values, x * df['delta'].values**2])
    coef, *_ = np.linalg.lstsq(X, df['dIV'].values, rcond=None)
    a, b, c = map(float, coef)
    corr_atm = a + b * 0.5 + c * 0.25
    print(f'  HW-TS estimate ({min(years)}–{max(years)}, n={len(df):,}): '
          f'a={a:+.4f} b={b:+.4f} c={c:+.4f} | δ=0.5 correction factor at={corr_atm:+.4f} '
          f'({"δ_MV<δ_BS ✓consistent with HW findings on SPX" if corr_atm < 0 else "⚠ positive sign, opposite to the HW experience, record it"})')
    return a, b, c

def _nonov_idx(eps_df, min_days=7):
    ds = pd.to_datetime(eps_df['date_start']).to_numpy()
    sel, last = [], None
    for i in np.argsort(ds):
        if last is None or (ds[i] - last) / np.timedelta64(1, 'D') >= min_days:
            sel.append(i); last = ds[i]
    return np.array(sorted(sel), dtype=int)

def _bb(x_dict, stat, B=3000, L=5, seed=0):
    n = len(next(iter(x_dict.values()))); rng = np.random.default_rng(seed)
    nb_ = int(np.ceil(n / L)); out = np.empty(B)
    for b in range(B):
        st = rng.integers(0, n, nb_)
        idx = np.concatenate([(s0 + np.arange(L)) % n for s0 in st])[:n]
        out[b] = stat({k: np.asarray(v)[idx] for k, v in x_dict.items()})
    lo, hi = np.percentile(out, [2.5, 97.5])
    return float(lo), float(hi), float((out <= 0).mean())

print('baseline machinery ready (HW-TS / WW band / replay / bootstrap)')

# ─────

In [ ]:
# ── PART D. classic baseline (E0convention): HW-TS / WW vs BS; resid pairing uses what PART B saved to disk dp ──
BL_ROWS = []
for RG, TY, VY, TEY in REGIMES:
    switch_regime(RG)
    eps_te = load_eps('test')
    dates = _ep_dates(eps_te)
    sp0 = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_te.iterrows()])
    pnl_bs, _ = _replay_pos(eps_te, _pos_bs, 2)
    a, b, c = estimate_hw_ts(list(TRAIN_YEARS) + list(VAL_YEARS))
    pnl_hw, _ = _replay_pos(eps_te, make_hw_fn(a, b, c), 2)
    dp_hw = (pnl_hw - pnl_bs) / sp0 * 1e4
    r = summarize_dp(dp_hw, dates, tag=f'[{RG}] HW-MV'); r.update(regime=RG, arm='HW-MV', kappa=0)
    BL_ROWS.append(r)
    eps_va = load_eps('validation')
    for kappa in KAPPAS:
        pnl_bs_va, rew_bs_va = _replay_pos(eps_va, _pos_bs, kappa)
        best_g, best_obj = None, -1e18
        for gam in WW_GAMMA_GRID:
            _, rew_va = _replay_pos(eps_va, make_ww_fn(gam), kappa)
            obj = float((rew_va - rew_bs_va).mean())
            if obj > best_obj: best_obj, best_g = obj, gam
        pnl_ww, _ = _replay_pos(eps_te, make_ww_fn(best_g), kappa)
        dp_ww = (pnl_ww - pnl_bs) / sp0 * 1e4
        r = summarize_dp(dp_ww, dates, tag=f'[{RG}] WW(γ*={best_g}) κ={kappa}')
        r.update(regime=RG, arm=f'WW(γ*={best_g})', kappa=kappa); BL_ROWS.append(r)
        if kappa == 2: dp_ww_k2 = dp_ww
    dp_res = np.load(f'{UNI_DIR}/dp_{RG}_k2.npy')
    for tag2, ref in [('resid−HW', dp_hw), ('resid−WW', dp_ww_k2)]:
        d_ = dp_res - ref
        m, lo, hi, p = full_stat_ci(d_, dates, np.mean)
        print(f'[{RG}] {tag2} (κ=2): Δ={m:+.2f}bp [{lo:+.2f},{hi:+.2f}] p_le0={p:.3f}')
        BL_ROWS.append({'regime': RG, 'arm': tag2, 'kappa': 2,
                        'full': m, 'full_lo': lo, 'full_hi': hi, 'full_p': p})
    print()
BL = pd.DataFrame(BL_ROWS)
BL_DIR = f'{DRIVE_REPO}/checkpoints/baselines_v3'; os.makedirs(BL_DIR, exist_ok=True)
BL.to_csv(f'{BL_DIR}/baselines_v3.csv', index=False)
print('saved →', BL_DIR)

## Reproducibility checks


### Repeat a selected training run


In [ ]:
# ── PART E1. sentinel recheck: 3config×2retrains, assert parameters are bitwise equal ──
import hashlib
SENTINELS = [('crash_test2018', 1, 0), ('mix_test2022', 3, 4), ('mix_test2024', 2, 2)]
for rg, kp, sd in SENTINELS:
    switch_regime(rg)
    ptr = pack_episodes(load_eps('train'))
    hs = []
    for rep in range(2):
        pol = train_residual(ptr, kp, best_beta[kp], ALPHA, EPOCHS, LR, seed=sd)
        blob = b''.join(p.detach().cpu().numpy().tobytes() for p in pol.parameters())
        hs.append(hashlib.md5(blob).hexdigest()[:12])
    ok = hs[0] == hs[1]
    print(f'sentinel {rg} κ={kp} s={sd}: {hs[0]} vs {hs[1]} → {"PASS" if ok else "FAIL"}')
    assert ok, 'within-session determinism is broken!'
print('✓ all sentinel rechecks passed (bitwise deterministic within a session)')

### Compare episode-level results across runs


In [ ]:
# Check that saved episode arrays and their CSV exports agree.
for regime, _, _, _ in REGIMES:
    for kappa in KAPPAS:
        saved = np.load(f'{UNI_DIR}/dp_{regime}_k{kappa}.npy')
        exported = pd.read_csv(f'{UNI_DIR}/perep_dp_{regime}_k{kappa}.csv')['dp_bp'].to_numpy()
        np.testing.assert_allclose(saved, exported, rtol=1e-12, atol=1e-12)
print('Episode array exports verified.')


### Save the experiment manifest


In [ ]:
# ── PART E3. manifest: the paper's sole citation list ──
import glob, platform, json as _json
import numpy as _np, scipy as _sc, pandas as _pdv, torch as _t
DEST = f'{DRIVE_REPO}/checkpoints/consolidated_v2'; os.makedirs(DEST, exist_ok=True)
pats = (glob.glob(f'{UNI_DIR}/*.csv') + glob.glob(f'{UNI_DIR}/*.npy')
        + glob.glob(f'{DRIVE_REPO}/checkpoints/baselines_v3/*.csv')
        + glob.glob(f'{EXP_DIR}/*.csv') + glob.glob(f'{EXP_DIR}/*.pkl'))
for rg, _, _, _ in REGIMES:
    pats += glob.glob(f'{DRIVE_REPO}/checkpoints/regime_{rg}/regime_best_beta.json')
    pats += glob.glob(f'{DRIVE_REPO}/checkpoints/regime_{rg}/regime_search.csv')
man = {'env': f'py={platform.python_version()} cpu={os.cpu_count()} thr={_t.get_num_threads()} '
              f'np={_np.__version__} sp={_sc.__version__} pd={_pdv.__version__} torch={_t.__version__}',
       'parquet_md5': _file_md5(CLEAN_PARQUET), 'files': {}}
import shutil
for p in sorted(set(pats)):
    dst = os.path.join(DEST, os.path.basename(p))
    shutil.copy(p, dst); man['files'][os.path.basename(p)] = _file_md5(p)
with open(f'{DEST}/manifest.json', 'w') as fh: _json.dump(man, fh, indent=1, ensure_ascii=False)
print(f'Saved {len(man["files"])} files and checksums to {DEST}.')

In [ ]:
# ── F. absolute tail comparison (safety claim): CVaR5(policyown P&L) vs CVaR5(BS own P&L), E0convention ──
import numpy as np, torch

def paired_tail_ci(strat, bs, dates, B=3000, block_days=10, seed=0):
    """paired day blocksbootstrap: computed on the same batch of resampled days CVaR5(policy)−CVaR5(BS)."""
    d_uniq, inv = np.unique(dates, return_inverse=True)
    groups = [np.where(inv == k)[0] for k in range(len(d_uniq))]
    rng = np.random.default_rng(seed)
    n = len(d_uniq); nb = int(np.ceil(n / block_days))
    point = cvar5(strat) - cvar5(bs)
    out = np.empty(B)
    for b_ in range(B):
        starts = rng.integers(0, n, nb)
        di = np.concatenate([(s + np.arange(block_days)) % n for s in starts])[:n]
        sel_ = np.concatenate([groups[k] for k in di])
        out[b_] = cvar5(strat[sel_]) - cvar5(bs[sel_])
    lo, hi = np.percentile(out, [2.5, 97.5])
    return float(point), float(lo), float(hi), float((out <= 0).mean())

F_ROWS = []
for RG, TY, VY, TEY in REGIMES:
    switch_regime(RG)
    eps_te = load_eps('test'); pte = pack_episodes(eps_te)
    dates = _ep_dates(eps_te)
    sp0 = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in eps_te.iterrows()])
    bs_bp = _episode_pnl(pte, pte['b_pos']).detach().cpu().numpy() / sp0 * 1e4
    for kappa in KAPPAS:
        dp = np.load(f'{UNI_DIR}/dp_{RG}_k{kappa}.npy')
        st_bp = bs_bp + dp
        t_s, t_b = cvar5(st_bp), cvar5(bs_bp)
        d_, lo, hi, p = paired_tail_ci(st_bp, bs_bp, dates)
        rel = d_ / abs(t_b) * 100 if abs(t_b) > 1e-9 else np.nan
        verdict = '✓improvement' if lo > 0 else ('≈no degradation' if hi > 0 else '⚠worsens')
        print(f'[{RG} κ={kappa}] absolute tail: policy={t_s:+.1f}bp  BS={t_b:+.1f}bp  '
              f'Δ={d_:+.2f} [{lo:+.2f},{hi:+.2f}]  ({rel:+.1f}% of |BStail|)  {verdict}')
        F_ROWS.append({'regime': RG, 'kappa': kappa, 'tail_strat': t_s, 'tail_bs': t_b,
                       'd_tail': d_, 'lo': lo, 'hi': hi, 'p_le0': p, 'rel_pct': rel})
    print()
F = pd.DataFrame(F_ROWS)
F.to_csv(f'{UNI_DIR}/abs_tail_comparison.csv', index=False)
n_ok = int((F['hi'] > 0).sum())
print(f'aggregate: {n_ok}/12 cells whose CI touches or crosses 0 → "absolute tail does not worsen"holds in these cells')
print(f'saved → {UNI_DIR}/abs_tail_comparison.csv  (E3 manifest auto-included)')